# MiniMAE-ViT End-to-End Tutorial

This notebook demonstrates the full `minimae_vit` system:

1. Build Hydra-style config in notebook
2. Auto-download CIFAR-10 and build dataloaders
3. Run MAE self-supervised pretraining
4. Visualize **pixel-space** reconstructions from normalized predictions
5. Run linear probing (frozen encoder)
6. Run full finetuning (train encoder + classifier)
7. Export encoder-only artifact (`vit_encoder_backbone.pt`)
8. Run inference over an image folder using the packaged `infer.py` script


> **Runtime note:** Defaults here are intentionally small (`epochs=1`, `max_steps` set) so the notebook runs as a practical tutorial. Increase values for stronger results.


In [ ]:
from __future__ import annotations

import json
import os
import random
import shutil
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from omegaconf import OmegaConf
from torchvision.utils import make_grid

from minimae_vit.data.datamodules import build_dataloaders
from minimae_vit.models.mae import MAE
from minimae_vit.models.patching import patchify, unpatchify
from minimae_vit.train.loops import pretrain_mae, train_classifier
from minimae_vit.utils.checkpoint import load_checkpoint
from minimae_vit.utils.seed import seed_everything


In [ ]:
seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch:", torch.__version__)
print("device:", device)


## 1) Build a complete config (Hydra-compatible shape)


In [ ]:
cfg = OmegaConf.create(
    {
        "seed": 42,
        "output_root": "outputs",
        "run_name": "notebook_e2e_demo",
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "compile": False,
        "ddp": False,
        "amp_dtype": "fp16",
        "pretrained_ckpt": None,
        "classifier_ckpt": None,
        "image_dir": None,
        "dataset": {
            "name": "cifar10",
            "root": "data",
            "img_size": 64,
            "num_classes": 10,
            "batch_size": 64,
            "num_workers": 2,
            "pin_memory": True,
            "pretrain_augment": True,
        },
        "model": {
            "img_size": 64,
            "patch_size": 8,
            "in_chans": 3,
            "embed_dim": 192,
            "depth": 4,
            "num_heads": 3,
            "mlp_ratio": 4.0,
            "drop": 0.0,
            "attn_drop": 0.0,
            "mask_ratio": 0.75,
            "dec_embed_dim": 128,
            "dec_depth": 2,
            "dec_heads": 4,
            "norm_eps": 1e-6,
        },
        "train": {
            "task": "mae",
            "epochs": 1,
            "lr": 3e-4,
            "weight_decay": 0.05,
            "warmup_epochs": 0,
            "log_interval": 10,
            "save_every": 1,
            "smoke_test": True,
            "max_steps": 30,
        },
        "logging": {
            "jsonl": True,
            "tensorboard": True,
            "wandb": False,
            "project": "minimae-vit",
        },
    }
)

print(OmegaConf.to_yaml(cfg))


## 2) Data loading (auto-download CIFAR-10)


In [ ]:
dataloaders = build_dataloaders(cfg)
print({k: len(v) for k, v in dataloaders.items()})

xb, yb = next(iter(dataloaders["train"]))
print("train batch:", xb.shape, yb.shape)

plt.figure(figsize=(7, 7))
plt.axis("off")
plt.title("CIFAR-10 samples (upsampled to 64x64)")
plt.imshow(make_grid(xb[:36], nrow=6).permute(1, 2, 0))
plt.show()


## 3) MAE pretraining (self-supervised)


In [ ]:
if Path(cfg.output_root, cfg.run_name).exists():
    shutil.rmtree(Path(cfg.output_root, cfg.run_name))

mae = MAE(cfg.model)
out_dir = pretrain_mae(cfg, mae, dataloaders)
print("Pretraining outputs:", out_dir)

ckpt_last = Path(out_dir) / "checkpoints" / "last.pt"
print("last checkpoint exists:", ckpt_last.exists(), ckpt_last)


## 4) Pixel-space reconstruction visualization

The MAE predicts normalized patch targets. To visualize correctly in pixel space, we unnormalize with per-patch `mean` and `var` from `aux`.


In [ ]:
@torch.no_grad()
def reconstruct_pixel_space(mae_model: MAE, imgs: torch.Tensor):
    mae_model.eval()
    loss, pred_norm, mask, aux = mae_model(imgs)

    patch_mean = aux["patch_mean"]
    patch_var = aux["patch_var"]
    eps = mae_model.cfg.norm_eps

    target = patchify(imgs, patch=mae_model.cfg.patch_size)
    pred_pixels = pred_norm * torch.sqrt(patch_var + eps) + patch_mean

    masked_target = target.clone()
    masked_target[mask.bool()] = 0.0

    img_orig = imgs
    img_masked = unpatchify(masked_target, patch=mae_model.cfg.patch_size, img_size=mae_model.cfg.img_size)
    img_pred = unpatchify(pred_pixels, patch=mae_model.cfg.patch_size, img_size=mae_model.cfg.img_size)

    # keep visible patches from original, masked patches from prediction
    blended = target.clone()
    blended[mask.bool()] = pred_pixels[mask.bool()]
    img_recon = unpatchify(blended, patch=mae_model.cfg.patch_size, img_size=mae_model.cfg.img_size)

    return float(loss.item()), img_orig.clamp(0, 1), img_masked.clamp(0, 1), img_pred.clamp(0, 1), img_recon.clamp(0, 1)

mae = MAE(cfg.model)
ckpt = load_checkpoint(ckpt_last)
mae.load_state_dict(ckpt["model_state"])
mae.to(device)

x_vis, _ = next(iter(dataloaders["test"]))
x_vis = x_vis[:16].to(device)
loss_val, img_orig, img_masked, img_pred, img_recon = reconstruct_pixel_space(mae, x_vis)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(make_grid(img_orig.cpu(), nrow=4).permute(1, 2, 0)); axes[0].set_title("Original")
axes[1].imshow(make_grid(img_masked.cpu(), nrow=4).permute(1, 2, 0)); axes[1].set_title("Masked input")
axes[2].imshow(make_grid(img_pred.cpu(), nrow=4).permute(1, 2, 0)); axes[2].set_title("Predicted patches")
axes[3].imshow(make_grid(img_recon.cpu(), nrow=4).permute(1, 2, 0)); axes[3].set_title("Reconstruction")
for ax in axes:
    ax.axis("off")
plt.suptitle(f"MAE masked loss: {loss_val:.4f}")
plt.tight_layout()
plt.show()


## 5) Linear probe (freeze encoder)


In [ ]:
cfg_lp = OmegaConf.create(OmegaConf.to_container(cfg, resolve=True))
cfg_lp.run_name = "notebook_linear_probe"
cfg_lp.train.task = "linear_probe"
cfg_lp.train.epochs = 1
cfg_lp.train.lr = 1e-3
cfg_lp.train.weight_decay = 0.0

mae_lp = MAE(cfg_lp.model)
mae_lp.load_state_dict(load_checkpoint(ckpt_last)["model_state"])
out_lp, best_lp = train_classifier(cfg_lp, mae_lp.encoder, dataloaders, freeze_encoder=True)
print("linear probe output:", out_lp)
print("linear probe best acc:", best_lp)


## 6) Full finetuning (encoder + classifier)


In [ ]:
cfg_ft = OmegaConf.create(OmegaConf.to_container(cfg, resolve=True))
cfg_ft.run_name = "notebook_finetune"
cfg_ft.train.task = "finetune"
cfg_ft.train.epochs = 1
cfg_ft.train.lr = 3e-4
cfg_ft.train.weight_decay = 0.05

mae_ft = MAE(cfg_ft.model)
mae_ft.load_state_dict(load_checkpoint(ckpt_last)["model_state"])
out_ft, best_ft = train_classifier(cfg_ft, mae_ft.encoder, dataloaders, freeze_encoder=False)
print("finetune output:", out_ft)
print("finetune best acc:", best_ft)


## 7) Export encoder artifact


In [ ]:
encoder_export_path = Path(cfg.output_root) / "notebook_exports" / "vit_encoder_backbone.pt"
encoder_export_path.parent.mkdir(parents=True, exist_ok=True)

mae_export = MAE(cfg.model)
mae_export.load_state_dict(load_checkpoint(ckpt_last)["model_state"])

payload = {
    "cfg": OmegaConf.to_container(cfg, resolve=True),
    "encoder_state_dict": mae_export.encoder.state_dict(),
}
torch.save(payload, encoder_export_path)
print("saved:", encoder_export_path)


## 8) Folder inference demo using `scripts/infer.py`


In [ ]:
# Prepare a few sample images from CIFAR-10 test set
sample_dir = Path("outputs/notebook_infer_images")
sample_dir.mkdir(parents=True, exist_ok=True)

x_test, _ = next(iter(dataloaders["test"]))
for i in range(8):
    img = x_test[i]
    p = sample_dir / f"sample_{i}.png"
    torch_img = (img.clamp(0, 1) * 255).byte().permute(1, 2, 0).cpu().numpy()
    from PIL import Image
    Image.fromarray(torch_img).save(p)

best_classifier = Path(out_ft) / "checkpoints" / "best.pt"
cmd = [
    "python",
    "scripts/infer.py",
    f"classifier_ckpt={best_classifier}",
    f"image_dir={sample_dir}",
]
print("Running:", " ".join(str(c) for c in cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print("stdout:\n", result.stdout)
print("stderr:\n", result.stderr)
print("return code:", result.returncode)


## 9) What to run next

- Increase MAE epochs and remove `max_steps` for stronger representations.
- Switch to `dataset.name=stl10` for larger unlabeled pretraining.
- Launch TensorBoard:

```bash
tensorboard --logdir outputs
```
